# 🧥 TRYON AI — CatVTON GPU Worker (Google Colab T4)

This notebook hosts the official **CatVTON (Concatenation Is All You Need for Virtual Try-On)** model on a free NVIDIA Tesla T4 GPU in Google Colab, exposing a secure HTTPS API via Cloudflare Tunnel for the TRYON AI backend.

> ⚠️ **Important Google Colab Notice**:
> *Google Colab Free is a development/testing environment. GPU availability, session duration, disconnects, and resource limits are controlled by Google and may change. This worker is not a reliable 24/7 production backend.*

### Pipeline Architecture:
**Android App** ➔ **FastAPI Backend** ➔ **CatVTONWorkerClient** ➔ **HTTPS Cloudflare Tunnel** ➔ **This Colab T4 Worker** ➔ **CatVTON Model**

---

In [ ]:
# ==============================================================================
# CELL 1 — Environment & NVIDIA GPU Verification
# ==============================================================================
import sys
import os
import shutil

print(f"Python Version: {sys.version}")

# Check PyTorch and CUDA
try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        raise RuntimeError("❌ CRITICAL: No NVIDIA GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")
    
    gpu_name = torch.cuda.get_device_name(0)
    vram_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cuda_ver = torch.version.cuda
    print(f"GPU Model: {gpu_name}")
    print(f"Total VRAM: {vram_total_gb:.2f} GB")
    print(f"CUDA Version: {cuda_ver}")

    if vram_total_gb < 11.0:
        print(f"⚠️ Warning: Detected {vram_total_gb:.2f} GB VRAM. CatVTON runs best with >= 12GB (Tesla T4).")
    else:
        print("✅ GPU Memory Check: PASSED (Tesla T4 15-16GB ready).")
except Exception as e:
    raise RuntimeError(f"GPU Verification Failed: {e}")

# Disk space check
total_b, used_b, free_b = shutil.disk_usage("/")
print(f"Available Disk Space: {free_b / (1024**3):.1f} GB free of {total_b / (1024**3):.1f} GB")
if free_b / (1024**3) < 15.0:
    print("⚠️ Warning: Free disk space is under 15GB. Clean up unneeded files before model download.")
else:
    print("✅ Disk Space Check: PASSED.")


In [ ]:
# ==============================================================================
# CELL 2 — Install Required Dependencies & Cloudflared Tunnel
# ==============================================================================
!pip install -q \
    "fastapi>=0.109.0" \
    "uvicorn[standard]>=0.27.0" \
    "python-multipart>=0.0.7" \
    "httpx>=0.26.0" \
    "pillow>=10.0.0" \
    "diffusers>=0.29.0" \
    "transformers>=4.40.0" \
    "accelerate>=0.30.0" \
    "einops>=0.7.0" \
    "safetensors>=0.4.0" \
    "huggingface_hub>=0.23.0" \
    "torchvision"

# Install Cloudflare Tunnel (cloudflared) binary for secure HTTPS tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!cloudflared --version

print("✅ Dependencies and Cloudflare Tunnel installed successfully!")


In [ ]:
# ==============================================================================
# CELL 3 — Clone Official CatVTON Repository
# ==============================================================================
# Official Source: https://github.com/Zheng-Chong/CatVTON
# Accepted at ICLR 2025 by Chong Zheng et al.
import os

if not os.path.exists("/content/CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON
    print("✅ Official CatVTON repository cloned.")
else:
    print("✅ CatVTON repository already present.")

# Verify core pipeline file exists
assert os.path.exists("/content/CatVTON/model/pipeline.py"), "❌ Missing CatVTON/model/pipeline.py"
print("✅ Pipeline files verified.")


In [ ]:
# ==============================================================================
# CELL 4 — Download Required Model Weights from Hugging Face
# ==============================================================================
# Model Weights Source: https://huggingface.co/zhengchong/CatVTON
# Base Inpainting Model: runwayml/stable-diffusion-inpainting (or booksforcharlie)
# License: CC-BY-NC-SA 4.0 (Non-commercial research / prototype)
from huggingface_hub import snapshot_download
import os

weights_dir = "/content/models/CatVTON"
os.makedirs(weights_dir, exist_ok=True)

print("Downloading CatVTON attention weights from Hugging Face Hub (zhengchong/CatVTON)...")
snapshot_download(
    repo_id="zhengchong/CatVTON",
    local_dir=weights_dir,
    allow_patterns=["*.json", "*mix-48k-1024*", "*.safetensors", "*.bin"],
    ignore_patterns=["*.msgpack", "*flux*"]
)
print("✅ CatVTON attention weights downloaded successfully!")


In [ ]:
# ==============================================================================
# CELL 5 — Verify Model Files on Disk
# ==============================================================================
import os

print("Checking downloaded model files:")
found_files = []
for root, dirs, files in os.walk("/content/models/CatVTON"):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        found_files.append((f, size_mb))
        print(f" - {f} ({size_mb:.2f} MB)")

assert len(found_files) > 0, "❌ No model files found in /content/models/CatVTON!"
print(f"✅ Model verification complete: {len(found_files)} files confirmed.")


In [ ]:
# ==============================================================================
# CELL 6 — Load CatVTON Pipeline into GPU (FP16 / T4 Memory Optimized)
# ==============================================================================
import sys
import time
import torch

if "/content/CatVTON" not in sys.path:
    sys.path.append("/content/CatVTON")

from model.pipeline import CatVTONPipeline

start_t = time.time()
print("Loading CatVTON Pipeline in FP16 precision on CUDA...")

try:
    pipeline = CatVTONPipeline(
        base_ckpt="booksforcharlie/stable-diffusion-inpainting",
        attn_ckpt="/content/models/CatVTON",
        attn_ckpt_version="mix",
        weight_dtype=torch.float16,
        device="cuda",
        use_tf32=True
    )
    print(f"✅ Pipeline successfully loaded into GPU memory in {time.time() - start_t:.1f}s!")
    alloc_mem = torch.cuda.memory_allocated(0) / (1024 ** 3)
    print(f"Current GPU Allocated Memory: {alloc_mem:.2f} GB")
except Exception as e:
    print(f"❌ Failed to load CatVTON pipeline: {e}")
    raise e


In [ ]:
# ==============================================================================
# CELL 7 — Local Test Inference (Mandatory Validation Before Exposing API)
# ==============================================================================
import os
import time
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display

# Prepare test directory and images
test_dir = "/content/tryon_worker/test"
os.makedirs(test_dir, exist_ok=True)

# Create a synthetic person portrait (768x1024)
p_test = Image.new("RGB", (768, 1024), color=(240, 240, 245))
d_p = ImageDraw.Draw(p_test)
d_p.ellipse([300, 100, 468, 280], fill=(230, 195, 175)) # Head
d_p.polygon([(260, 280), (508, 280), (620, 680), (148, 680)], fill=(70, 75, 90)) # Torso
d_p.rectangle([200, 680, 568, 980], fill=(40, 45, 55)) # Lower
p_test.save(f"{test_dir}/test_person.jpg")

# Create a synthetic garment (red blazer)
g_test = Image.new("RGB", (768, 1024), color=(255, 255, 255))
d_g = ImageDraw.Draw(g_test)
d_g.polygon([(240, 220), (528, 220), (640, 660), (128, 660)], fill=(185, 40, 45)) # Blazer
g_test.save(f"{test_dir}/test_garment.jpg")

# Generate Torso Mask
mask_test = Image.new("L", (768, 1024), 0)
d_m = ImageDraw.Draw(mask_test)
d_m.polygon([(240, 220), (528, 220), (640, 660), (128, 660)], fill=255)
mask_test = mask_test.filter(ImageFilter.GaussianBlur(8))

print("Running local test inference on NVIDIA T4 GPU...")
t0 = time.time()
with torch.inference_mode():
    results = pipeline(
        image=p_test,
        condition_image=g_test,
        mask=mask_test,
        num_inference_steps=30,
        guidance_scale=2.5,
        seed=42
    )
test_result_img = results[0] if isinstance(results, list) else results

                # Preserve original face and facial identity with zero alteration
                face_h = int(1024 * 0.22)
                test_result_img.paste(p_test.crop((0, 0, 768, face_h)), (0, 0))
                elapsed = time.time() - t0

output_test_path = f"{test_dir}/local_test_result.jpg"
test_result_img.save(output_test_path, format="JPEG", quality=95)

peak_vram = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
print(f"✅ Local inference SUCCESS! Elapsed: {elapsed:.2f}s | Peak VRAM: {peak_vram:.2f} GB")
print("Displaying generated test image:")
display(test_result_img.resize((384, 512)))


In [ ]:
# ==============================================================================
# CELL 8 — Write and Start the FastAPI Worker Server
# ==============================================================================
import os
import subprocess
import time
import getpass

# Generate or enter secret API key
default_key = "tryon_secret_" + os.urandom(6).hex()
print("Enter worker API key (press Enter to use auto-generated secure token):")
user_key = input(f"[{default_key}]: ").strip()
WORKER_API_KEY = user_key if user_key else default_key
os.environ["CATVTON_WORKER_API_KEY"] = WORKER_API_KEY
os.environ["TRYON_WORKER_DIR"] = "/content/tryon_worker"

# Launch Uvicorn in the background on port 8000
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "worker.worker_server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("/content/worker_server.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

# Test health endpoint locally
!curl -s http://localhost:8000/health
print("\n✅ FastAPI Worker Server is running on port 8000!")


In [ ]:
# ==============================================================================
# CELL 9 — Start Secure Cloudflare Tunnel (HTTPS)
# ==============================================================================
import subprocess
import re
import time

tunnel_log = "/content/cloudflared.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)

print("Starting Cloudflare HTTPS tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT
)

# Wait and parse public URL
public_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if not public_url:
    print("❌ Failed to obtain Cloudflare public URL. Inspect log:")
    !cat /content/cloudflared.log | tail -n 20
    raise RuntimeError("Cloudflare tunnel failed to start.")

print(f"✅ Secure Cloudflare Tunnel active: {public_url}")


In [ ]:
# ==============================================================================
# CELL 10 — Display Worker URL & Backend Integration Instructions
# ==============================================================================
print("=" * 78)
print("🚀 CATVTON GPU WORKER READY FOR TRYON AI BACKEND!")
print("=" * 78)
print(f"Worker Public URL : {public_url}")
print(f"Worker API Key    : {WORKER_API_KEY}")
print("-" * 78)
print("Copy and paste these settings into your TRYON AI backend .env:")
print("-" * 78)
print(f"CATVTON_WORKER_URL={public_url}")
print(f"CATVTON_WORKER_API_KEY={WORKER_API_KEY}")
print("VTON_PROVIDER=catvton")
print("DEMO_MODE=false")
print("AUTO_FALLBACK_TO_DEMO=true")
print("=" * 78)

# Test public URL through the tunnel
!curl -s -H "Authorization: Bearer $WORKER_API_KEY" {public_url}/health
print("\n✅ End-to-end HTTPS endpoint verified!")
